# Week 5 - BTK simulations and domain gap

Render DES-like griz blends from CatSim. Keep the simulated test partition fixed and use the training and validation partitions for any calibration.

In [ ]:
%pip install 'numpy<2' 'scipy<1.16' 'astropy<7' 'photutils==2.3.0' 'pandas<3' 'pyarrow<22' huggingface_hub blending-toolkit==1.0.9 surveycodex==1.3.0 galsim==2.8.5

In [ ]:
# Importing necessary libraries
import json
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

REPO = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "Week4").exists())
W4, W5 = REPO / "Week4", REPO / "Week5"
sys.path[:0] = [str(W4 / "src"), str(W5 / "src")]


In [ ]:
# Downloading the tutorial CatSim catalogue
from urllib.request import urlretrieve
from simulations import generate

catalog = W5 / "data/input_catalog.fits"
manifest = json.loads((W5 / "data/simulation_manifest.json").read_text())
url = ("https://raw.githubusercontent.com/LSSTDESC/BlendingToolKit/"
       f"{manifest['catalog_revision']}/{manifest['catalog_path']}")
if not catalog.exists():
    urlretrieve(url, catalog)
sim = W5 / "data/sim"
for name, config in manifest["splits"].items():
    generate(catalog, sim / name, name, config["count"], config["seed"],
             jobs=manifest["workers"])


In [ ]:
# Inspecting simulated RGB, galaxy ownership and semantic masks
from matplotlib.colors import ListedColormap
cmap = ListedColormap(["black", "gold", "royalblue"])
examples = sorted((sim / "validation").glob("*.npz"))[:6]
fig, axes = plt.subplots(len(examples), 3, figsize=(8, 2.5 * len(examples)), squeeze=False)
for path, row in zip(examples, axes):
    with np.load(path) as sample:
        row[0].imshow(sample["rgb"])
        row[1].imshow(sample["instance"], cmap="tab10", vmin=0, vmax=9)
        row[2].imshow(sample["semantic"], cmap=cmap, vmin=0, vmax=2)
    for ax, title in zip(row, (path.stem, "Instance IDs", "Semantic mask")):
        ax.set_title(title)
        ax.axis("off")
fig.tight_layout()
fig.savefig(W5 / "results/domain_gap/sim_examples.png", dpi=120)
plt.show()


In [ ]:
# Loading the saved split
splits = json.loads((W4 / "data/splits.json").read_text())
def real_records(ids):
    excluded = set()
    for filename in ("automatic_exclusions.json", "excluded_ids.json"):
        path = W4 / "data" / filename
        if path.exists():
            excluded.update(json.loads(path.read_text()))
    ids = [key for key in ids if key not in excluded]
    return [(key, W4 / f"data/cutouts/{key}.npz", W4 / f"results/pseudo_labels/{key}.npy")
            for key in ids]


In [ ]:
# Comparing noise, seeing, flux S/N and target size on development data
from domain import compare_domains
real = [W4 / f"data/cutouts/{key}.npz" for key in splits["train"]]
synthetic = sorted((sim / "train").glob("*.npz"))
output = W5 / "results/domain_gap"
compare_domains(real, synthetic, output)


In [ ]:
# Visualizing frozen CLS embeddings
from sklearn.manifold import TSNE
from heads import backbone, preprocess
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
model = backbone("dinov2_vits14", DEVICE)
rng = np.random.default_rng(42)
paths = list(rng.choice(real, 200, replace=False)) + list(rng.choice(synthetic, 200, replace=False))
vectors = []
for start in range(0, len(paths), 16):
    images = []
    for path in paths[start:start + 16]:
        with np.load(path) as sample:
            images.append(sample["rgb"])
    x = preprocess(torch.from_numpy(np.stack(images)).permute(0, 3, 1, 2), 224, DEVICE)
    with torch.no_grad():
        vectors.extend(model.forward_features(x)["x_norm_clstoken"].cpu().numpy())
xy = TSNE(random_state=42, init="pca", learning_rate="auto").fit_transform(np.asarray(vectors))
plt.scatter(*xy[:200].T, s=10, label="Real")
plt.scatter(*xy[200:].T, s=10, label="BTK")
plt.legend()
plt.savefig(output / "cls_tsne.png", dpi=150, bbox_inches="tight")
plt.show()
